# 03 · Wrażliwość na parametry

Trzy parametry CHAMELEON-a wpływają na wynik:

- **`k_nn`** — rozmiar sąsiedztwa w grafie wstępnym,
- **`alpha`** — waga `relative_closeness` w score merge'a (`RI · RC^α`),
- **`min_cluster_size`** — minimalny rozmiar sub-klastra w Fazie I.

Każdy sweep agreguje wynik po 4 datasetach Karypisa z ground-truth (DS1, DS3, DS4, DS5) — linia to **mean**, wstęga to **±1 std**. Algorytm jest deterministyczny, więc rozrzut pochodzi tylko z różnic między datasetami: **wąska wstęga = parametr działa podobnie na wszystkich kształtach, szeroka = różne datasety wolą różne wartości**.

Każdy parametr ma 3 panele: ARI, NMI, oraz syntetyczna **Quality = (ARI + NMI) / 2** (jak w nb 01).

In [ ]:
import sys, os, importlib
sys.path.insert(0, os.getcwd())
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import _helpers as H
importlib.reload(H)

plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 120
pd.options.display.float_format = '{:.3f}'.format

## 1. Wpływ `k_nn`

`k_nn` ustala lokalność grafu. Za małe — graf rozspójniony, klastry rozdrobnione. Za duże — krawędzie międzyklastrowe, mergery nadmierne.

In [ ]:
def plot_sweep_trend(param, title=None, datasets=None):
    """Sweep zagregowany przez 4 datasety z GT: mean ±1 std dla każdej metryki.

    3 panele obok siebie: ARI, NMI, Quality = (ARI + NMI) / 2. Algorytm jest
    deterministyczny — zmienność na wykresie pochodzi WYŁĄCZNIE z różnic
    między datasetami, nie z runów.

    Y-zoom ustawiamy do najniższego (mean − std) z marginesem 5pp, żeby
    wzmocnić widoczność trendu zamiast topić go w pustym górnym zakresie.
    """
    if datasets is None:
        datasets = H.DATASETS_WITH_GT
    df = H.load_sweep(param)
    df = df[df['dataset'].isin(datasets)].copy()
    df['quality'] = (df['ari'] + df['nmi']) / 2

    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4), sharex=True)
    metric_specs = [
        ('ari', 'ARI', '#4C72B0'),
        ('nmi', 'NMI', '#DD8452'),
        ('quality', 'Quality = (ARI + NMI) / 2', '#55A868'),
    ]
    # Wspólne Y-limity między panelami: najniższy (mean−std) ze wszystkich
    # metryk, żeby panele były wzajemnie porównywalne wzrokowo.
    y_lo = min(
        (df.groupby('value')[col].mean() - df.groupby('value')[col].std()).min()
        for col, *_ in metric_specs
    )
    y_lo = max(0.0, y_lo - 0.05)
    y_hi = 1.02

    for ax, (col, label, color) in zip(axes, metric_specs):
        agg = df.groupby('value')[col].agg(['mean', 'std']).reset_index()
        ax.plot(agg['value'], agg['mean'], marker='o', color=color, linewidth=2,
                label=f'mean ({len(datasets)} dat.)')
        ax.fill_between(agg['value'],
                         agg['mean'] - agg['std'],
                         agg['mean'] + agg['std'],
                         color=color, alpha=0.18, label='±1 std')
        ax.set_xlabel(param)
        ax.set_ylabel(label)
        ax.set_title(label)
        ax.set_ylim(y_lo, y_hi)
        ax.grid(True, linestyle=':', alpha=0.4)
        ax.legend(fontsize=8, loc='lower right')
        # Etykiety X dokladnie na zmierzonych wartosciach (zamiast auto-tickow,
        # ktore dla mcs potrafia naklada "0.025" / "0.030").
        xvals = agg['value'].values
        ax.set_xticks(xvals)
        # Rotujemy przy wartosciach ulamkowych albo gdy etykiet jest duzo.
        needs_rotation = len(xvals) > 5 or any(
            isinstance(v, float) and not float(v).is_integer() for v in xvals
        )
        if needs_rotation:
            ax.tick_params(axis='x', labelrotation=35)
            for lbl in ax.get_xticklabels():
                lbl.set_horizontalalignment('right')
    if title:
        fig.suptitle(title, y=1.02, fontsize=12)
    plt.tight_layout()
    plt.show()

plot_sweep_trend('k_nn', title='Trend wrażliwości na k_nn (mean ±1 std po DS1/DS3/DS4/DS5)')

## 2. Wpływ `alpha`

`alpha` = wykładnik dla `relative_closeness` w score merge'a. Paper Karypis 1999 zaleca `alpha ∈ {1.5, 2.0, 2.5, 3.0}`. Sweep rozszerzamy do `{0.5, 1.0, …, 3.0}`.

In [ ]:
plot_sweep_trend('alpha', title='Trend wrażliwości na alpha (mean ±1 std po DS1/DS3/DS4/DS5)')

## 3. Wpływ `min_cluster_size`

Wartości jako frakcja `n` (`0.01 = 1 %`, `0.05 = 5 %`).

In [ ]:
plot_sweep_trend('min_cluster_size',
                 title='Trend wrażliwości na min_cluster_size (mean ±1 std po DS1/DS3/DS4/DS5)')

**Co widać po agregacji:**

- **`k_nn`**: **wyraźny monotoniczny trend rosnący**
- **`alpha`**: **wyraźny monotoniczny trend rosnący**
- **`min_cluster_size`**: **monotoniczny spadek**

## 4. Najlepsze parametry per dataset (rezultat HPO)

Z `results/hpo_best.json` — siatka grid-search po (k_nn × alpha × min_cluster_size), maksymalizacja ARI vs ground-truth.

In [ ]:
hpo = pd.DataFrame(H.load_hpo_best()).T
hpo.index.name = 'dataset'
hpo = hpo[['n_clusters', 'k_nn', 'alpha', 'min_cluster_size', 'ari', 'nmi', 'runtime_s']]
hpo.style.format({'ari': '{:.3f}', 'nmi': '{:.3f}', 'runtime_s': '{:.3f}'})

## 5. Wnioski

1. **`alpha`** ma najczystszy trend agregowany — ARI i NMI rosną monotonicznie z `α`, a rozrzut między datasetami się zmniejsza. Wysokie `α` (2.5-3.0) to bezpieczna heurystyka, jeśli nie robi się per-dataset HPO.
2. **`min_cluster_size`** powyżej 1-2% `n` → monotoniczny spadek jakości (potwierdzony na ARI i NMI). Trzymać `mcs` blisko dolnej granicy.
3. **`k_nn`** — sweet spot 10-30, słaby trend w tym zakresie. Wyraźny tylko anty-trend dla `k=5` (graf zbyt rzadki).